Clustering methods to be used:
- KMEans
- HDBScan

In [ ]:
# Installing requirements

import pandas as pd
import numpy as np

!pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer

from sklearn.cluster import KMeans

!pip install hdbscan
import hdbscan

from sklearn.preprocessing import normalize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

In [ ]:
file_path = "/content/corpus.tsv"

df = pd.read_csv(file_path, sep="\t", header=None)
df.columns = ["text", "split", "label"]

# Preview
df.head()

,text,split,label
0,broadband ahead join internet fast accord offi...,train,tech
1,plan share sale owner technology dominate inde...,train,business
2,mobile rack mobile phone celebrate anniversary...,train,tech
3,launch reconstruction drive appeal peace natio...,train,business
4,buy giant profit soar acquisition big firm tax...,train,business


In [ ]:
texts = df["text"].astype(str).tolist()

print(texts[:3])

['broadband ahead join internet fast accord official figure number business connect jump report broadband connection end compare nation rank world telecom body election campaign ensure affordable high speed net access american accord report broadband increasingly popular research shopping download music watch video total number business broadband rise end compare hook broadband subscriber line technology ordinary phone line support high data speed cable lead account line broadband phone line connection accord figure', 'plan share sale owner technology dominate index plan sell share public list market operate accord document file stock market plan raise sale observer step close full public icon technology boom recently pour cold water suggestion company sell share private technically public stock start trade list equity trade money sale investor buy share private filing document share technology firm company high growth potential symbol internet telecom boom bubble burst recovery fortun

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(embeddings.shape)

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

(2225, 384)


In [ ]:
embeddings_norm = normalize(embeddings, norm="l2") #Normalization for similarity calculations

Labels to evaluate

In [ ]:
true_labels = df["label"]

In [ ]:
true_labels.unique()

array(['tech', 'business', 'entertainment', 'sport', 'politics'],
      dtype=object)

In [ ]:
le = LabelEncoder()
true_labels_encoded = le.fit_transform(true_labels)

print(list(le.classes_))  #see label mapping

['business', 'entertainment', 'politics', 'sport', 'tech']


**KMeans**

In [ ]:
k = 5

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(embeddings_norm)

print(kmeans_labels[:10])

[1 4 1 4 4 0 3 0 4 1]


In [ ]:
ari = adjusted_rand_score(true_labels_encoded, kmeans_labels)
nmi = normalized_mutual_info_score(true_labels_encoded, kmeans_labels)
sil = silhouette_score(embeddings_norm, kmeans_labels)

print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")
print(f"Silhouette Score: {sil:.4f}")

ARI: 0.8659
NMI: 0.8310
Silhouette Score: 0.0822


**HDBScan**

In [ ]:
hdb = hdbscan.HDBSCAN(
    min_cluster_size=50,
    min_samples=7,
    metric='euclidean'
)

hdb_labels = hdb.fit_predict(embeddings_norm)

print(hdb_labels[:20])

[-1 -1 -1  0 -1 -1  1 -1  0 -1 -1 -1  0 -1 -1 -1 -1 -1 -1 -1]


In [ ]:
noise_ratio = sum(hdb_labels == -1) / len(hdb_labels)
print(noise_ratio)

0.7892134831460674


In [ ]:
mask = hdb_labels != -1

filtered_labels = hdb_labels[mask]
filtered_true = true_labels_encoded[mask]
filtered_embeddings = embeddings_norm[mask]

print("Remaining points:", len(filtered_labels))

Remaining points: 469


In [ ]:
print("Unique labels:", np.unique(hdb_labels))
print("Noise points:", sum(hdb_labels == -1))
print("Num clusters:", len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0))

Unique labels: [-1  0  1]
Noise points: 1756
Num clusters: 2


In [ ]:
ari_hdb = adjusted_rand_score(filtered_true, filtered_labels)
nmi_hdb = normalized_mutual_info_score(filtered_true, filtered_labels)

# Silhouette only if >1 cluster
if len(set(filtered_labels)) > 1:
    sil_hdb = silhouette_score(filtered_embeddings, filtered_labels)
else:
    sil_hdb = -1

print(f"HDBSCAN ARI: {ari_hdb:.4f}")
print(f"HDBSCAN NMI: {nmi_hdb:.4f}")
print(f"HDBSCAN Silhouette: {sil_hdb:.4f}")

HDBSCAN ARI: 0.3268
HDBSCAN NMI: 0.5450
HDBSCAN Silhouette: 0.1374


**GAUSSIAN MIXTURE MODEL**